# 05b. Мутационные спектры четырёх предсказанных классов

Ноутбук 05 сравнивает два спектра, заданных порогом T95 одноклассового
Isolation Forest. Здесь та же нормировка применяется к четырём классам тяжести,
предсказанным моделью из ноутбука 02b.

## Почему группировка строится на модели без частотного признака

Спектр взвешивается числом носителей варианта. Если бы принадлежность к классу
определялась в том числе популяционной частотой, то группы различались бы по той
самой величине, которая затем задаёт вес, и различие спектров было бы отчасти
следствием построения, а не свойством мутационного процесса.

Поэтому группировка берётся из
[`scripts/spectrum_grouping.py`](../scripts/spectrum_grouping.py): та же модель
переобучается на панели без `hom_rarity_soft` — локальное ограничение,
консервативность, класс последствия и положение в кодоне. Принадлежность к
классу тогда независима от весов.

Цена этого решения измерена и составляет заметную часть качества: macro F1
падает с 0.53 до 0.41, квадратично взвешенная каппа с 0.71 до 0.39. Взамен
различие между спектрами классов поддаётся интерпретации.

Тот же довод применим и к разметке T95 из ноутбука 05: Isolation Forest был
обучен на четырёх признаках редкости, поэтому существующее бинарное сравнение
несёт ту же циркулярность в неизмеренном виде.

## Что изменено относительно ноутбука 05

Ориентация цепи, расчёт возможностей, общий знаменатель и позиционный бутстреп
не изменены. Отличается только колонка группировки, число групп и построение
графиков: вместо одной пары сравниваются четыре класса, а разности приводятся
для трёх контрастов относительно `benign`.

Общий знаменатель охватывает все четыре группы, поэтому частоты всех
4 × 192 = 768 ячеек в сумме дают единицу, а доля каждой группы отражает её вклад
в общую спектральную массу.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../scripts").resolve()))
from mutspec192_ci import (
    SBS192_REFERENCE_ORDER,
    estimate_shared_spectrum_uncertainty,
    plot_mutspec192_difference_with_ci,
    plot_mutspec192_with_ci,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

In [ ]:
# ============================================================
# Paths and deterministic 192-component order
# ============================================================

INPUT_PATH = Path("../results/spectrum_groups/spectrum_groups_with_weights_192_T95.tsv")
MITOMAP_LOCI_INPUT = Path("../data/raw/mitomap/mitomap_genome_loci_192_r889.tsv")

OUTPUT_DIR = Path("../results/mutation_spectra_by_class")
FIGURE_DIR = Path("../results/figures/mutation_spectra_by_class_192")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_INPUT = Path("../results/classification/spectrum_class_assignment.tsv")
SPECTRUM_OUTPUT = OUTPUT_DIR / "spectrum_weighted_frequencies_192_by_class.tsv"
UNCERTAINTY_OUTPUT = OUTPUT_DIR / "spectrum_uncertainty_192_by_class.tsv"
DIFFERENCE_UNCERTAINTY_OUTPUT = (
    OUTPUT_DIR / "spectrum_difference_uncertainty_192_by_class.tsv"
)
N_BOOTSTRAP_SIMULATIONS = 4000
BOOTSTRAP_RANDOM_SEED = 20260810

WEIGHT_COL = "combined_db_spectrum_weight"
GROUP_COL = "spectrum_class"
GROUP_ORDER = ["benign", "vus_low", "vus_high", "pathogenic"]
EXPECTED_COMPLEMENT_LOCI = {
    "MT-ND6", "MT-TQ", "MT-TA", "MT-TN", "MT-TC",
    "MT-TY", "MT-TS1", "MT-TE", "MT-TP",
}
DNA_COMPLEMENT = {"A": "T", "C": "G", "G": "C", "T": "A"}

substitution_order_12 = [
    "A>C", "A>G", "A>T",
    "C>A", "C>G", "C>T",
    "G>A", "G>C", "G>T",
    "T>A", "T>C", "T>G",
]
substitution_order_192 = [
    f"{left}[{substitution}]{right}"
    for left in "ACGT"
    for substitution in substitution_order_12
    for right in "ACGT"
]

assert len(substitution_order_192) == 192
assert len(set(substitution_order_192)) == 192

In [ ]:
df = pd.read_csv(INPUT_PATH, sep="\t", low_memory=False)

# Grouping comes from the frequency-free model, so class membership is
# independent of the carrier counts that weight the spectrum.
class_assignment = pd.read_csv(
    CLASS_INPUT, sep="\t", usecols=["variant_id", GROUP_COL]
)
n_before = len(df)
df = df.merge(class_assignment, on="variant_id", how="left")
if len(df) != n_before:
    raise ValueError("Class merge changed the row count")
if df[GROUP_COL].isna().any():
    raise ValueError("Some variants have no predicted class")
loci = pd.read_csv(MITOMAP_LOCI_INPUT, sep="\t")

required_cols = {
    "variant_id", "position", "reference", "alternate",
    "left_base", "right_base", "trinucleotide_context",
    "substitution_type_12", "substitution_type_192",
    "is_valid_snv_substitution", "combined_db_observed",
    "combined_db_count_raw", "combined_db_count_pc",
    "combined_db_total_records", "ref_context_count",
    "combined_db_spectrum_weight",
    "combined_db_spectrum_weight_raw", GROUP_COL,
}
missing_cols = sorted(required_cols.difference(df.columns))
if missing_cols:
    raise ValueError(f"Missing required 192-spectrum columns: {missing_cols}")

complement_loci = loci[loci["map_locus"].isin(EXPECTED_COMPLEMENT_LOCI)].copy()
observed_complement_loci = set(
    complement_loci.loc[
        complement_loci["description"].str.contains(
            "[on complement]", regex=False, na=False
        ),
        "map_locus",
    ]
)
if observed_complement_loci != EXPECTED_COMPLEMENT_LOCI:
    raise ValueError(
        f"Unexpected complement-strand loci: {sorted(observed_complement_loci)}"
    )

def expand_circular_interval(start, end, genome_length=16569):
    start = int(start)
    end = int(end)
    if start <= end:
        return range(start, end + 1)
    return list(range(start, genome_length + 1)) + list(range(1, end + 1))


complement_positions = set()
for row in complement_loci.itertuples(index=False):
    complement_positions.update(expand_circular_interval(row.start, row.end))

work = df[df["is_valid_snv_substitution"] == 1].copy()
rcrs_cols = [
    "reference", "alternate", "left_base", "right_base",
    "trinucleotide_context", "substitution_type_12",
    "substitution_type_192", "ref_context_count",
    "combined_db_spectrum_weight",
    "combined_db_spectrum_weight_raw",
]
for column in rcrs_cols:
    work[f"{column}_rcrs"] = work[column]

work["is_complement_strand"] = work["position"].isin(complement_positions)
complement_mask = work["is_complement_strand"]
work.loc[complement_mask, "left_base"] = (
    work.loc[complement_mask, "right_base_rcrs"].map(DNA_COMPLEMENT)
)
work.loc[complement_mask, "right_base"] = (
    work.loc[complement_mask, "left_base_rcrs"].map(DNA_COMPLEMENT)
)
work.loc[complement_mask, "reference"] = (
    work.loc[complement_mask, "reference_rcrs"].map(DNA_COMPLEMENT)
)
work.loc[complement_mask, "alternate"] = (
    work.loc[complement_mask, "alternate_rcrs"].map(DNA_COMPLEMENT)
)
work["trinucleotide_context"] = (
    work["left_base"] + work["reference"] + work["right_base"]
)
work["substitution_type_12"] = work["reference"] + ">" + work["alternate"]
work["substitution_type_192"] = (
    work["left_base"] + "[" + work["substitution_type_12"]
    + "]" + work["right_base"]
)

position_universe = (
    work[["position", "trinucleotide_context", "is_complement_strand"]]
    .drop_duplicates("position")
)
if len(position_universe) * 3 != len(work):
    raise ValueError("Expected exactly three alternate SNVs per position.")
context_counts = position_universe["trinucleotide_context"].value_counts()
work["ref_context_count"] = work["trinucleotide_context"].map(context_counts)

rcrs_position_universe = (
    work[["position", "trinucleotide_context_rcrs"]]
    .drop_duplicates("position")
)
rcrs_context_counts = (
    rcrs_position_universe["trinucleotide_context_rcrs"].value_counts()
)
if not np.array_equal(
    work["ref_context_count_rcrs"].to_numpy(),
    work["trinucleotide_context_rcrs"].map(rcrs_context_counts).to_numpy(),
):
    raise ValueError("rCRS context opportunities no longer match notebook 04.")

db_total = pd.to_numeric(work["combined_db_total_records"], errors="raise")
if db_total.nunique() != 1 or (db_total <= 0).any():
    raise ValueError("Expected one positive combined database denominator.")
work[WEIGHT_COL] = (
    work["combined_db_count_pc"] / (db_total * work["ref_context_count"])
)
work["combined_db_spectrum_weight_raw"] = (
    work["combined_db_count_raw"] / (db_total * work["ref_context_count"])
)

if int(position_universe["is_complement_strand"].sum()) != len(complement_positions):
    raise ValueError("Not every complement-strand position was oriented.")
if not work.loc[~complement_mask, "substitution_type_192"].equals(
    work.loc[~complement_mask, "substitution_type_192_rcrs"]
):
    raise ValueError("Reference-strand categories changed unexpectedly.")
nd6_position_14149 = work[work["position"] == 14149]
observed_nd6_categories = dict(zip(
    nd6_position_14149["alternate_rcrs"],
    nd6_position_14149["substitution_type_192"],
))
expected_nd6_categories = {
    "A": "G[G>T]T", "G": "G[G>C]T", "T": "G[G>A]T",
}
if observed_nd6_categories != expected_nd6_categories:
    raise ValueError(
        f"Unexpected strand-oriented ND6 example: {observed_nd6_categories}"
    )

work = work[work[GROUP_COL] != "exclude"].copy()

unknown_categories = sorted(
    set(work["substitution_type_192"].dropna()) - set(substitution_order_192)
)
if unknown_categories:
    raise ValueError(f"Unknown 192-component categories: {unknown_categories[:10]}")

print("Input shape:", df.shape)
print("Analysis table shape:", work.shape)
print("Complement-strand loci:", sorted(observed_complement_loci))
print("Reverse-complemented positions:", len(complement_positions))
print("Categories represented:", work["substitution_type_192"].nunique(), "/ 192")
print("\nGroups:")
print(work[GROUP_COL].value_counts(dropna=False))
print("\nWeight summary:")
print(work[WEIGHT_COL].describe())

In [ ]:
# Adapt the top-variant QC from notebook 04 to each comparison group.
observed_category_counts = (
    work.loc[work["combined_db_observed"] == 1]
    .groupby(GROUP_COL)["substitution_type_192"]
    .nunique()
    .rename("n_observed_categories")
)

variant_contributors = work.assign(
    within_group_weight_share=(
        work[WEIGHT_COL]
        / work.groupby(GROUP_COL)[WEIGHT_COL].transform("sum")
    )
).sort_values(
    [GROUP_COL, "within_group_weight_share"],
    ascending=[True, False],
)
variant_contributors["rank_within_group"] = (
    variant_contributors.groupby(GROUP_COL).cumcount() + 1
)
variant_contributors["cumulative_group_weight_share"] = (
    variant_contributors.groupby(GROUP_COL)["within_group_weight_share"].cumsum()
)

variant_concentration = (
    variant_contributors.groupby(GROUP_COL)["within_group_weight_share"]
    .agg(
        top_1_variant_weight_share=lambda shares: shares.nlargest(1).sum(),
        top_5_variant_weight_share=lambda shares: shares.nlargest(5).sum(),
        top_10_variant_weight_share=lambda shares: shares.nlargest(10).sum(),
    )
)

category_contributors = (
    work.groupby([GROUP_COL, "substitution_type_192"])[WEIGHT_COL]
    .sum()
    .rename("category_weight")
    .to_frame()
)
category_contributors["within_group_weight_share"] = (
    category_contributors["category_weight"]
    / category_contributors.groupby(level=0)["category_weight"].transform("sum")
)
top_10_category_share = (
    category_contributors.groupby(level=0)["within_group_weight_share"]
    .agg(lambda shares: shares.nlargest(10).sum())
    .rename("top_10_category_weight_share")
)

support_summary = (
    work.groupby(GROUP_COL)
    .agg(
        n_variants=("variant_id", "size"),
        n_observed_variants=("combined_db_observed", "sum"),
        n_categories=("substitution_type_192", "nunique"),
        total_group_weight=(WEIGHT_COL, "sum"),
    )
    .join(observed_category_counts)
    .join(variant_concentration)
    .join(top_10_category_share)
    .reset_index()
)
support_summary["observed_variant_fraction"] = (
    support_summary["n_observed_variants"] / support_summary["n_variants"]
)

summary_share_cols = [
    "observed_variant_fraction",
    "top_1_variant_weight_share",
    "top_5_variant_weight_share",
    "top_10_variant_weight_share",
    "top_10_category_weight_share",
]
support_summary_display = support_summary.copy()
for col in summary_share_cols:
    support_summary_display[col] = support_summary_display[col].map("{:.2%}".format)
support_summary_display["total_group_weight"] = (
    support_summary_display["total_group_weight"].map("{:.6g}".format)
)
display(support_summary_display)

top_variant_contributors = (
    variant_contributors.groupby(GROUP_COL, group_keys=False)
    .head(10)
    [[
        GROUP_COL,
        "rank_within_group",
        "variant_id",
        "is_complement_strand",
        "substitution_type_192",
        "substitution_type_192_rcrs",
        "combined_db_count_raw",
        "ref_context_count",
        WEIGHT_COL,
        "within_group_weight_share",
        "cumulative_group_weight_share",
    ]]
)
top_variant_contributors_display = top_variant_contributors.copy()
top_variant_contributors_display["combined_db_count_raw"] = (
    top_variant_contributors_display["combined_db_count_raw"].map("{:,.0f}".format)
)
top_variant_contributors_display[WEIGHT_COL] = (
    top_variant_contributors_display[WEIGHT_COL].map("{:.3e}".format)
)
for col in ["within_group_weight_share", "cumulative_group_weight_share"]:
    top_variant_contributors_display[col] = (
        top_variant_contributors_display[col].map("{:.2%}".format)
    )
display(top_variant_contributors_display)

In [ ]:
def compute_weighted_spectrum(data, group_col, weight_col):
    spectrum = (
        data
        .groupby([group_col, "substitution_type_192"], as_index=False)
        .agg(
            n_variants=("variant_id", "count"),
            weighted_sum=(weight_col, "sum"),
        )
        .rename(columns={
            group_col: "group_name",
            "substitution_type_192": "substitution_type",
        })
    )

    all_groups = sorted(spectrum["group_name"].dropna().unique())
    complete_index = pd.MultiIndex.from_product(
        [all_groups, substitution_order_192],
        names=["group_name", "substitution_type"],
    )
    spectrum = (
        spectrum
        .set_index(["group_name", "substitution_type"])
        .reindex(complete_index)
        .reset_index()
    )
    spectrum["n_variants"] = spectrum["n_variants"].fillna(0).astype(int)
    spectrum["weighted_sum"] = spectrum["weighted_sum"].fillna(0.0)
    spectrum["total_weight"] = spectrum["weighted_sum"].sum()
    if spectrum["total_weight"].iloc[0] <= 0:
        raise ValueError("The shared spectrum total must be positive.")
    spectrum["weighted_frequency"] = spectrum["weighted_sum"] / spectrum["total_weight"]
    spectrum.insert(0, "group_col", group_col)
    spectrum["weight_col"] = weight_col
    return spectrum


spectrum = compute_weighted_spectrum(work, GROUP_COL, WEIGHT_COL)
spectrum.to_csv(SPECTRUM_OUTPUT, sep="\t", index=False)

assert spectrum.groupby("group_name").size().eq(192).all()
assert np.isclose(spectrum["weighted_frequency"].sum(), 1.0)

spectrum_uncertainty, spectrum_difference_uncertainty = (
    estimate_shared_spectrum_uncertainty(
        work,
        group_col=GROUP_COL,
        group_order=GROUP_ORDER,
        category_col="substitution_type_192",
        category_order=SBS192_REFERENCE_ORDER,
        count_col="combined_db_count_pc",
        opportunity_col="ref_context_count",
        position_col="position",
        n_simulations=N_BOOTSTRAP_SIMULATIONS,
        random_seed=BOOTSTRAP_RANDOM_SEED,
    )
)
spectrum_uncertainty.to_csv(UNCERTAINTY_OUTPUT, sep="\t", index=False)
spectrum_difference_uncertainty.to_csv(
    DIFFERENCE_UNCERTAINTY_OUTPUT, sep="\t", index=False
)

point_qc = (
    spectrum[["group_name", "substitution_type", "weighted_frequency"]]
    .merge(
        spectrum_uncertainty.rename(columns={
            GROUP_COL: "group_name",
            "substitution_type_192": "substitution_type",
            "weighted_frequency": "bootstrap_point_frequency",
        })[[
            "group_name", "substitution_type",
            "bootstrap_point_frequency",
        ]],
        on=["group_name", "substitution_type"],
        how="inner",
        validate="one_to_one",
    )
)
if len(point_qc) != len(GROUP_ORDER) * 192 or not np.allclose(
    point_qc["weighted_frequency"],
    point_qc["bootstrap_point_frequency"],
):
    raise ValueError("Bootstrap point spectrum does not match the main spectrum.")

print("Saved:", SPECTRUM_OUTPUT)
print("Saved:", UNCERTAINTY_OUTPUT)
print("Saved:", DIFFERENCE_UNCERTAINTY_OUTPUT)
print("FDR-significant SBS192 differences:", int(
    spectrum_difference_uncertainty["significant_fdr_05"].sum()
))
display(spectrum.head(24))

In [ ]:
group_label_map = {
    "benign": "Benign-like",
    "vus_low": "VUS, low significance",
    "vus_high": "VUS, high significance",
    "pathogenic": "Pathogenic-like",
    "expanded_neutral_like_T95": "Expanded neutral-like",
    "strict_expanded_neutral_like_T95": "Strict expanded neutral-like",
    "unlabeled_out_of_neutral_domain_T95": "Out-of-neutral-domain",
    "known_neutral_reference": "Known neutral reference",
    "article_pathogenic_posthoc": "Article pathogenic post-hoc",
    "disease_suspected_posthoc": "Disease suspected post-hoc",
    "unlabeled_neutral_like_T95": "Unlabeled neutral-like",
}


def safe_filename(text):
    return (
        str(text)
        .replace("/", "_")
        .replace(" ", "_")
        .replace(">", "to")
        .replace(":", "_")
    )


def plot_single_spectrum(spectrum_df, group_name, output_dir=FIGURE_DIR):
    plot_df = (
        spectrum_df[spectrum_df["group_name"] == group_name]
        .set_index("substitution_type")
        .reindex(substitution_order_192)
        .reset_index()
    )

    x = np.arange(len(substitution_order_192))
    y = plot_df["weighted_frequency"].values

    label = group_label_map.get(group_name, group_name)
    n_variants = int(plot_df["n_variants"].sum())

    plt.figure(figsize=(32, 7))
    plt.bar(x, y)
    plt.xticks(x, substitution_order_192, rotation=90, ha="center", fontsize=5)
    plt.ylabel("Shared-normalized DB-weighted frequency")
    plt.xlabel("Functional-strand-oriented trinucleotide substitution")
    plt.title(f"{label} n={n_variants}")
    plt.tight_layout()

    output_path = output_dir / f"spectrum_{safe_filename(group_name)}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return output_path


def plot_two_group_comparison(spectrum_df, group_1, group_2, output_dir=FIGURE_DIR):
    df1 = (
        spectrum_df[spectrum_df["group_name"] == group_1]
        .set_index("substitution_type")
        .reindex(substitution_order_192)
        .reset_index()
    )

    df2 = (
        spectrum_df[spectrum_df["group_name"] == group_2]
        .set_index("substitution_type")
        .reindex(substitution_order_192)
        .reset_index()
    )

    x = np.arange(len(substitution_order_192))
    width = 0.4

    label_1 = group_label_map.get(group_1, group_1)
    label_2 = group_label_map.get(group_2, group_2)

    plt.figure(figsize=(32, 7))
    plt.bar(x - width / 2, df1["weighted_frequency"].values, width=width, label=label_1)
    plt.bar(x + width / 2, df2["weighted_frequency"].values, width=width, label=label_2)
    plt.xticks(x, substitution_order_192, rotation=90, ha="center", fontsize=5)
    plt.ylabel("Shared-normalized DB-weighted frequency")
    plt.xlabel("Functional-strand-oriented trinucleotide substitution")
    plt.title("DB-weighted mutational spectrum comparison (shared denominator)")
    plt.legend()
    plt.tight_layout()

    output_path = output_dir / f"spectrum_comparison_{safe_filename(group_1)}_vs_{safe_filename(group_2)}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return output_path


def plot_spectrum_difference(spectrum_df, reference_group, comparison_group, output_dir=FIGURE_DIR):
    ref = (
        spectrum_df[spectrum_df["group_name"] == reference_group]
        .set_index("substitution_type")
        .reindex(substitution_order_192)
        .reset_index()
    )

    comp = (
        spectrum_df[spectrum_df["group_name"] == comparison_group]
        .set_index("substitution_type")
        .reindex(substitution_order_192)
        .reset_index()
    )

    diff = comp["weighted_frequency"].values - ref["weighted_frequency"].values

    x = np.arange(len(substitution_order_192))

    label_ref = group_label_map.get(reference_group, reference_group)
    label_comp = group_label_map.get(comparison_group, comparison_group)

    plt.figure(figsize=(32, 7))
    plt.bar(x, diff)
    plt.axhline(0, linewidth=1)
    plt.xticks(x, substitution_order_192, rotation=90, ha="center", fontsize=5)
    plt.ylabel("Difference in shared-normalized DB-weighted frequency")
    plt.xlabel("Functional-strand-oriented trinucleotide substitution")
    plt.title(f"{label_comp} minus {label_ref}")
    plt.tight_layout()

    output_path = output_dir / f"spectrum_difference_{safe_filename(comparison_group)}_minus_{safe_filename(reference_group)}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return output_path

In [ ]:
# Per-class spectra
for group_name in GROUP_ORDER:
    plot_mutspec192_with_ci(
        spectrum_uncertainty,
        group_order=[group_name],
        group_labels=group_label_map,
        group_col=GROUP_COL,
        category_col="substitution_type_192",
        title=group_label_map[group_name],
        ylabel="Shared-normalized DB-weighted frequency",
        output_path=FIGURE_DIR / f"spectrum_{safe_filename(group_name)}.png",
    )

# Contrasts against the benign-like class. The plotting helper draws at most two
# groups on one axis, so each contrast is a separate figure; the denominator
# still spans all four classes.
reference_group = GROUP_ORDER[0]

for comparison_group in GROUP_ORDER[1:]:
    pair_difference = spectrum_difference_uncertainty[
        (spectrum_difference_uncertainty["reference_group"] == reference_group)
        & (spectrum_difference_uncertainty["comparison_group"] == comparison_group)
    ].copy()
    if pair_difference.empty:
        raise ValueError(f"No difference rows for {reference_group} vs {comparison_group}")

    plot_mutspec192_with_ci(
        spectrum_uncertainty,
        group_order=[reference_group, comparison_group],
        group_labels=group_label_map,
        difference=pair_difference,
        group_col=GROUP_COL,
        category_col="substitution_type_192",
        title=(
            f"{group_label_map[reference_group]} vs "
            f"{group_label_map[comparison_group]} (denominator over all four classes)"
        ),
        ylabel="Shared-normalized DB-weighted frequency",
        output_path=(
            FIGURE_DIR
            / f"spectrum_comparison_{safe_filename(reference_group)}_vs_{safe_filename(comparison_group)}.png"
        ),
    )

    plot_mutspec192_difference_with_ci(
        pair_difference,
        category_col="substitution_type_192",
        title=(
            f"{group_label_map[comparison_group]} minus "
            f"{group_label_map[reference_group]}"
        ),
        output_path=(
            FIGURE_DIR
            / f"spectrum_difference_{safe_filename(comparison_group)}_minus_{safe_filename(reference_group)}.png"
        ),
    )

# How much of the shared spectral mass each class carries.
class_mass = (
    spectrum.groupby("group_name")["weighted_frequency"].sum()
    .reindex(GROUP_ORDER)
    .rename("share_of_shared_spectrum")
    .to_frame()
)
class_mass["n_variants"] = (
    spectrum.groupby("group_name")["n_variants"].sum().reindex(GROUP_ORDER)
)
class_mass["observed_channels"] = (
    spectrum[spectrum["n_variants"] > 0].groupby("group_name").size().reindex(GROUP_ORDER)
)
class_mass.to_csv(OUTPUT_DIR / "class_spectrum_mass.tsv", sep="\t")

significant = (
    spectrum_difference_uncertainty.groupby(
        ["reference_group", "comparison_group"]
    )["significant_fdr_05"].sum().rename("n_significant_channels").to_frame()
)
significant.to_csv(OUTPUT_DIR / "class_difference_significance.tsv", sep="\t")

print("Figures saved to:", FIGURE_DIR)
display(class_mass)
display(significant)


# How much of each class rests on variants nobody has observed. Those enter only
# through the pseudocount max(count, 1), so a channel built solely from them is
# not evidence of a mutational process.
observed_share = []
for group_name in GROUP_ORDER:
    block = work[work[GROUP_COL] == group_name]
    total = block[WEIGHT_COL].sum()
    unobserved = block.loc[block["combined_db_count_raw"] == 0, WEIGHT_COL].sum()
    channels_with_candidates = block["substitution_type_192"].nunique()
    channels_observed = (
        block.loc[block["combined_db_count_raw"] > 0, "substitution_type_192"].nunique()
    )
    observed_share.append({
        "group_name": group_name,
        "n_candidates": len(block),
        "n_unobserved": int((block["combined_db_count_raw"] == 0).sum()),
        "unobserved_weight_share": unobserved / total,
        "channels_with_candidates": channels_with_candidates,
        "channels_with_an_observed_variant": channels_observed,
    })
observed_share = pd.DataFrame(observed_share)
observed_share.to_csv(OUTPUT_DIR / "channel_observation_audit.tsv", sep="	", index=False)

display(observed_share.round(4))
